# AgriVox: YOLOv8n-cls 25-Class Agricultural Edge Diagnostic Model
### Full Training & INT8 TFLite Export Pipeline for Kerala Agricultural Crops

This notebook trains an ultra-lightweight **YOLOv8n-cls** model on **25 disease & healthy classes** covering:
- **Paddy / Rice (നെല്ല്)**: Rice Blast, Bacterial Leaf Blight, Healthy
- **Coconut (തെങ്ങ്)**: Bud Rot, Stem Bleeding, Healthy
- **Banana (വാഴ)**: Sigatoka Leaf Spot, Panama Wilt, Healthy
- **Brinjal (വഴുതന)**: Bacterial Wilt, Little Leaf, Healthy
- **Okra (വെണ്ട)**: Yellow Vein Mosaic Virus, Powdery Mildew, Healthy
- **Bell Pepper**: Bacterial Spot, Healthy
- **Tomato**: Early Blight, Late Blight, Leaf Mold, Septoria Leaf Spot, Healthy
- **Potato**: Early Blight, Late Blight, Healthy

**Export Target:** INT8 Quantized TensorFlow Lite (`model_quant.tflite`) for <40ms offline inference in AgriVox.

In [ ]:
# Step 1: Install & Upgrade Dependencies
!pip install -q --upgrade "kaggle>=2.2.2" ultralytics opencv-python-headless tensorflow
!kaggle --version


In [ ]:
# Step 2: Kaggle API Setup (Upload your kaggle.json)
import os
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your kaggle.json file (from Kaggle -> Settings -> Create New Token):")
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured successfully!")


In [ ]:
# Step 3: Dynamic Live Kaggle Search & Download
# Automatically searches Kaggle for public active datasets without hardcoded 403-restricted slugs
import os
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

os.makedirs("./raw_data", exist_ok=True)

# 1. Base Datasets (PlantVillage & Rice)
base_downloads = {
    "plantvillage": ["emmarex/plantdisease", "arjuntejaswi/plant-village"],
    "rice": ["vbookshelf/rice-leaf-diseases", "minhhuy2810/rice-diseases-image-dataset"]
}
for target_folder, slugs in base_downloads.items():
    dest = f"./raw_data/{target_folder}"
    os.makedirs(dest, exist_ok=True)
    if len(os.listdir(dest)) == 0:
        for s in slugs:
            try:
                print(f"Downloading {s} -> {dest}...")
                api.dataset_download_files(s, path=dest, unzip=True)
                if len(os.listdir(dest)) > 0:
                    print(f"  ✓ Successfully downloaded {s}")
                    break
            except Exception as e:
                print(f"  ✗ {s}: {e}")
    else:
        print(f"✓ {target_folder} already downloaded.")

# 2. Dynamic Live Search & Download for Kerala Crops (Banana, Coconut, Brinjal, Okra)
crops_to_search = {
    "banana": ["banana leaf disease", "banana disease"],
    "coconut": ["coconut leaf disease", "coconut disease", "coconut tree disease"],
    "brinjal": ["eggplant disease", "brinjal leaf disease", "brinjal disease"],
    "okra": ["okra leaf disease", "okra disease", "bhendi leaf disease"]
}

for crop_name, queries in crops_to_search.items():
    dest = f"./raw_data/{crop_name}"
    os.makedirs(dest, exist_ok=True)
    if len(os.listdir(dest)) > 0:
        print(f"✓ {crop_name} already downloaded ({len(os.listdir(dest))} items).")
        continue

    success = False
    for q in queries:
        print(f"Searching Kaggle for '{q}'...")
        try:
            results = api.dataset_list(search=q)
            for ds in results[:5]:
                ref = str(ds.ref)
                print(f"  Attempting: {ref}...")
                try:
                    api.dataset_download_files(ref, path=dest, unzip=True)
                    if len(os.listdir(dest)) > 0:
                        print(f"  ✓ Downloaded {ref} to {dest}!")
                        success = True
                        break
                except Exception as dl_e:
                    print(f"    Restricted/Error: {dl_e}")
            if success:
                break
        except Exception as search_e:
            print(f"  Search error: {search_e}")

print("\nAll available dataset downloads finished!")


In [ ]:
# Step 4: Dataset Directory Setup for YOLOv8 Classification
import os, shutil, random, glob

TARGET_ROOT = "./dataset"
if os.path.exists(TARGET_ROOT):
    shutil.rmtree(TARGET_ROOT)

AGRIVOX_CLASSES = [
    "Pepper_bell_Bacterial_spot", "Pepper_bell_healthy",
    "Potato_Early_blight", "Potato_Late_blight", "Potato_healthy",
    "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_healthy",
    "Rice_Blast", "Rice_Bacterial_Blight", "Rice_healthy",
    "Coconut_Bud_Rot", "Coconut_Stem_Bleeding", "Coconut_healthy",
    "Banana_Sigatoka_Leaf_Spot", "Banana_Panama_Wilt", "Banana_healthy",
    "Brinjal_Bacterial_Wilt", "Brinjal_Little_Leaf", "Brinjal_healthy",
    "Okra_Yellow_Vein_Mosaic", "Okra_Powdery_Mildew", "Okra_healthy"
]

for split in ["train", "val"]:
    for cls in AGRIVOX_CLASSES:
        os.makedirs(os.path.join(TARGET_ROOT, split, cls), exist_ok=True)

print(f"Created directory structure for {len(AGRIVOX_CLASSES)} classes.")


In [ ]:
# Step 5: Populate Train and Validation Splits (80/20 train/val)
random.seed(42)

def copy_class_images(src_patterns, target_class, max_imgs=600):
    images = []
    for pattern in src_patterns:
        images.extend(glob.glob(pattern, recursive=True))
    images = [img for img in images if img.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(images)
    selected = images[:max_imgs]
    split_idx = int(0.8 * len(selected))
    train_imgs = selected[:split_idx]
    val_imgs = selected[split_idx:]
    for img in train_imgs:
        shutil.copy(img, os.path.join(TARGET_ROOT, "train", target_class, os.path.basename(img)))
    for img in val_imgs:
        shutil.copy(img, os.path.join(TARGET_ROOT, "val", target_class, os.path.basename(img)))
    print(f"[{target_class}]: {len(train_imgs)} train, {len(val_imgs)} val (from {len(images)} found)")

# Solanaceae (PlantVillage)
pv_map = {
    "Pepper_bell_Bacterial_spot": ["./raw_data/plantvillage/**/Pepper*Bacterial*/**"],
    "Pepper_bell_healthy": ["./raw_data/plantvillage/**/Pepper*healthy*/**"],
    "Potato_Early_blight": ["./raw_data/plantvillage/**/Potato*Early*/**"],
    "Potato_Late_blight": ["./raw_data/plantvillage/**/Potato*Late*/**"],
    "Potato_healthy": ["./raw_data/plantvillage/**/Potato*healthy*/**"],
    "Tomato_Early_blight": ["./raw_data/plantvillage/**/Tomato*Early*/**"],
    "Tomato_Late_blight": ["./raw_data/plantvillage/**/Tomato*Late*/**"],
    "Tomato_Leaf_Mold": ["./raw_data/plantvillage/**/Tomato*Mold*/**"],
    "Tomato_Septoria_leaf_spot": ["./raw_data/plantvillage/**/Tomato*Septoria*/**"],
    "Tomato_healthy": ["./raw_data/plantvillage/**/Tomato*healthy*/**"],
}

# Rice
rice_map = {
    "Rice_Blast": ["./raw_data/rice/**/blast*/**", "./raw_data/rice/**/Blast*/**"],
    "Rice_Bacterial_Blight": ["./raw_data/rice/**/bacterial*/**", "./raw_data/rice/**/Bacterial*/**"],
    "Rice_healthy": ["./raw_data/rice/**/healthy*/**", "./raw_data/rice/**/Healthy*/**"],
}

# Banana
banana_map = {
    "Banana_Sigatoka_Leaf_Spot": ["./raw_data/banana/**/sigatoka*/**", "./raw_data/banana/**/Sigatoka*/**"],
    "Banana_Panama_Wilt": ["./raw_data/banana/**/panama*/**", "./raw_data/banana/**/wilt*/**"],
    "Banana_healthy": ["./raw_data/banana/**/healthy*/**", "./raw_data/banana/**/Healthy*/**"],
}

# Coconut, Brinjal, Okra
other_map = {
    "Coconut_Bud_Rot": ["./raw_data/coconut/**/bud_rot*/**", "./raw_data/coconut/**/Bud*/**"],
    "Coconut_Stem_Bleeding": ["./raw_data/coconut/**/stem*/**", "./raw_data/coconut/**/Bleeding*/**"],
    "Coconut_healthy": ["./raw_data/coconut/**/healthy*/**", "./raw_data/coconut/**/Healthy*/**"],
    "Brinjal_Bacterial_Wilt": ["./raw_data/brinjal/**/wilt*/**", "./raw_data/brinjal/**/bacterial*/**", "./raw_data/brinjal/**/disease*/**"],
    "Brinjal_Little_Leaf": ["./raw_data/brinjal/**/little*/**", "./raw_data/brinjal/**/Little*/**", "./raw_data/brinjal/**/leaf*/**"],
    "Brinjal_healthy": ["./raw_data/brinjal/**/healthy*/**", "./raw_data/brinjal/**/Healthy*/**"],
    "Okra_Yellow_Vein_Mosaic": ["./raw_data/okra/**/mosaic*/**", "./raw_data/okra/**/yellow*/**", "./raw_data/okra/**/disease*/**"],
    "Okra_Powdery_Mildew": ["./raw_data/okra/**/mildew*/**", "./raw_data/okra/**/powdery*/**"],
    "Okra_healthy": ["./raw_data/okra/**/healthy*/**", "./raw_data/okra/**/Healthy*/**"],
}

all_mappings = {**pv_map, **rice_map, **banana_map, **other_map}
for target_cls, patterns in all_mappings.items():
    copy_class_images(patterns, target_cls)

print("\nDataset curation complete!")


In [ ]:
# Step 6: Train YOLOv8n-cls Classification Model
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data="./dataset",
    epochs=25,
    imgsz=224,
    batch=32,
    workers=4,
    device=0,
    project="agrivox_training",
    name="yolov8n_25classes"
)
print("Model training complete!")


In [ ]:
# Step 7: Export to INT8 Quantized TensorFlow Lite (model_quant.tflite)
from ultralytics import YOLO

best_model = YOLO("agrivox_training/yolov8n_25classes/weights/best.pt")

exported_path = best_model.export(
    format="tflite",
    int8=True,
    imgsz=224,
    data="./dataset"
)
print(f"Exported TFLite model at: {exported_path}")


In [ ]:
# Step 8: Generate Standardized labels.txt & Download Model Bundle
import shutil, glob
from google.colab import files

classes = sorted(os.listdir("./dataset/train"))
with open("labels.txt", "w") as f:
    for cls in classes:
        f.write(cls + "\n")

print(f"Generated labels.txt ({len(classes)} classes):")
for i, cls in enumerate(classes):
    print(f"  [{i:02d}] {cls}")

tflite_candidates = (
    glob.glob("agrivox_training/yolov8n_25classes/weights/*int8*.tflite") +
    glob.glob("agrivox_training/yolov8n_25classes/weights/*.tflite")
)

if tflite_candidates:
    shutil.copy(tflite_candidates[0], "model_quant.tflite")
    print("\nDownload ready! Saving model_quant.tflite and labels.txt...")
    files.download("model_quant.tflite")
    files.download("labels.txt")
else:
    print("No .tflite file found in weights folder.")
